<a href="https://www.kaggle.com/code/william2020/apple-s-mlx-transformers-pe-multi-head-attn?scriptVersionId=187548653" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Apple's MLX: Input/Positional Embeddings and Multi-head Self-Attention for the Transformer architecture

#### Framework created by Apple.

In this tutorial, we will walk you through the core concepts and functionalities of MLX, starting with the basics of tokenization and positional encoding. Whether you’re a beginner looking to get started with machine learning on Apple hardware or an experienced practitioner seeking to optimize your workflows, this book provides practical examples and step-by-step instructions to help you harness the full potential of MLX.

In [4]:
!pip install -q mlx transformers

In [8]:
import numpy as np
import mlx.core as mx
import mlx.nn as nn
# from nltk.tokenize import word_tokenize
from collections import defaultdict
from typing import Optional, Union

from transformers import AutoTokenizer, AutoModelForCausalLM

hf_token="insert_here"


# Step 1: Tokenize the sentence

In [9]:
sentence = "What are the advantages and disadvantages of using a unified memory architecture?"
# Load the tokenizer with authentication
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    token=hf_token
)

tokens = tokenizer(sentence, return_tensors="pt")

In [10]:
print(tokens)

{'input_ids': tensor([[    1,  1724,   526,   278, 25486,   322,   766, 17263, 19771,   310,
           773,   263,   443,  2164,  3370, 11258, 29973]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


# Step 2: Create a simple vocabulary and convert tokens to indices

In [11]:
vocab = defaultdict(lambda: len(tokens))
indices = [vocab[token] for token in tokens]
print("Indices:", indices)

Indices: [2, 2]


# Step 3: Initialize the embedding layer

In [12]:
embedding_dim = 64
num_embeddings = len(vocab)
embedding_layer = nn.Embedding(num_embeddings, embedding_dim)
len(vocab)

2

# Step 4: Convert indices to MLX array and embed them

In [13]:
input_data = mx.array(indices)
embedded_tokens = embedding_layer(input_data)
print("Embedded Tokens:", embedded_tokens)

Embedded Tokens: array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=float32)


# RoPE (Rotary Positional Encoding)

RoPE applies a rotational transformation to the token embeddings based on their positions in the sequence. This transformation uses sinusoidal functions to create a set of rotation matrices that are applied to the embeddings. The result is a set of positionally encoded embeddings that carry rich relative positional information.

In transformer models, understanding the relative positions of tokens within a sequence is crucial for tasks that require contextual understanding, such as language modeling and translation.

# Calculate sequence length

In [14]:
seq_len = embedded_tokens.shape[0]
print("Sequence Length:", seq_len)

Sequence Length: 2


# Generate frequencies for the sinusoidal embeddings

In [15]:
inv_freq = 1.0 / (10000 ** (np.arange(0, embedding_dim, 2).astype(np.float32) / embedding_dim))
print("Inverse Frequencies:", inv_freq)

freqs = mx.array(np.outer(np.arange(seq_len), inv_freq).astype(np.float32))
print("Frequencies:", freqs)

Inverse Frequencies: [1.0000000e+00 7.4989420e-01 5.6234133e-01 4.2169651e-01 3.1622776e-01
 2.3713736e-01 1.7782794e-01 1.3335215e-01 1.0000000e-01 7.4989416e-02
 5.6234129e-02 4.2169649e-02 3.1622779e-02 2.3713736e-02 1.7782794e-02
 1.3335215e-02 9.9999998e-03 7.4989423e-03 5.6234132e-03 4.2169648e-03
 3.1622779e-03 2.3713738e-03 1.7782794e-03 1.3335214e-03 1.0000000e-03
 7.4989418e-04 5.6234130e-04 4.2169649e-04 3.1622779e-04 2.3713738e-04
 1.7782794e-04 1.3335215e-04]
Frequencies: array([[0, 0, 0, ..., 0, 0, 0],
       [1, 0.749894, 0.562341, ..., 0.000237137, 0.000177828, 0.000133352]], dtype=float32)


# Calculate cosine and sine of the frequencies

In [16]:
cos_pos = mx.cos(freqs)
sin_pos = mx.sin(freqs)
print("Cosine Positional Encoding:", cos_pos)
print("Sine Positional Encoding:", sin_pos)

Cosine Positional Encoding: array([[1, 1, 1, ..., 1, 1, 1],
       [0.540302, 0.731761, 0.846009, ..., 1, 1, 1]], dtype=float32)
Sine Positional Encoding: array([[0, 0, 0, ..., 0, 0, 0],
       [0.841471, 0.681561, 0.533168, ..., 0.000237137, 0.000177828, 0.000133352]], dtype=float32)


# Split embedded tokens into even and odd parts

 Splits the embedded tokens into even and odd parts.

In [17]:
x1 = embedded_tokens[:, ::2]
x2 = embedded_tokens[:, 1::2]
print("x1 (even):", x1.shape)
print("x2 (odd):", x2.shape)

x1 (even): (2, 32)
x2 (odd): (2, 32)


# Apply rotational transformation

Applies the rotational transformation to the split parts.

In [18]:
x1_new = x1 * cos_pos - x2 * sin_pos
x2_new = x1 * sin_pos + x2 * cos_pos
print("Transformed x1:", x1_new.shape)
print("Transformed x2:", x2_new.shape)

Transformed x1: (2, 32)
Transformed x2: (2, 32)


# Concatenate the new x1 and x2 back together

Concatenates the transformed parts back together to get the final positional encoded embeddings.

In [19]:
positional_encoded_embeddings = mx.concatenate([x1_new, x2_new], axis=-1)
print("Positional Encoded Embeddings:", positional_encoded_embeddings)

Positional Encoded Embeddings: array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=float32)


# Multi-Head Self-Attention

To enhance the model’s ability to capture different types of relationships, the transformer employs multi-head self-attention. This technique splits the Query, Key, and Value vectors into multiple smaller sub-vectors, each corresponding to a different attention head. The attention mechanism is applied independently to each head, and the results are concatenated and linearly transformed to produce the final output.


In [20]:
# Define the dimensions
num_heads = 8
head_dim = embedding_dim // num_heads

# Sample input embeddings after positional encoding (from previous section)
# For demonstration, we assume `positional_encoded_embeddings` is already defined
input_embeddings = positional_encoded_embeddings

# Add a batch dimension if missing
if len(input_embeddings.shape) == 2:
    input_embeddings = input_embeddings[np.newaxis, :, :]

print("Input Embeddings Shape:", input_embeddings.shape)

Input Embeddings Shape: (1, 2, 64)


# Linear Transformations

This cell applies linear projections to the input embeddings to obtain the Query, Key, and Value matrices.

In [21]:
# Step 1: Linear Transformations
query_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
key_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
value_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)

queries = query_proj(input_embeddings)
keys = key_proj(input_embeddings)
values = value_proj(input_embeddings)

print("Queries Shape:", queries.shape)
print("Keys Shape:", keys.shape)
print("Values Shape:", values.shape)

Queries Shape: (1, 2, 64)
Keys Shape: (1, 2, 64)
Values Shape: (1, 2, 64)


# Split for Multi-Head Attention

This cell reshapes the Query, Key, and Value matrices to prepare them for multi-head attention. It then transposes these matrices to separate the attention heads and prints their shapes.

In [22]:
batch_size, seq_length, _ = queries.shape

queries = queries.reshape(batch_size, seq_length, num_heads, head_dim).transpose(0, 2, 1, 3)
keys = keys.reshape(batch_size, seq_length, num_heads, head_dim).transpose(0, 2, 1, 3)
values = values.reshape(batch_size, seq_length, num_heads, head_dim).transpose(0, 2, 1, 3)

print("Split Queries Shape:", queries.shape)
print("Split Keys Shape:", keys.shape)
print("Split Values Shape:", values.shape)

Split Queries Shape: (1, 8, 2, 8)
Split Keys Shape: (1, 8, 2, 8)
Split Values Shape: (1, 8, 2, 8)


# Scaled Dot-Product Attention

This cell calculates the attention scores by performing scaled dot-product attention. It then applies the softmax function to obtain normalized attention weights and computes the final attention output.

In [23]:
# Step 3: Scaled Dot-Product Attention
dk = head_dim
scores = mx.matmul(queries, keys.transpose(0, 1, 3, 2)) / mx.sqrt(mx.array([dk], dtype=queries.dtype))
attention_weights = nn.softmax(scores, axis=-1)
attention_output = mx.matmul(attention_weights, values)

print("Attention Weights Shape:", attention_weights.shape)
print("Attention Output Shape:", attention_output.shape)

Attention Weights Shape: (1, 8, 2, 2)
Attention Output Shape: (1, 8, 2, 8)


# Combine Heads

This cell combines the output from all attention heads back into a single tensor. It reshapes the combined output to match the original embedding dimensions and prints the shape.

In [24]:
# Step 4: Combine Heads
combined_output = attention_output.transpose(0, 2, 1, 3).reshape(batch_size, seq_length, embedding_dim)
print("Combined Output Shape:", combined_output.shape)

Combined Output Shape: (1, 2, 64)


# Final Linear Transformation

This cell applies a final linear transformation to the combined output from the attention heads.

In [25]:
# Step 5: Final Linear Transformation
output_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
final_output = output_proj(combined_output)
print("Final Output after Self-Attention Shape:", final_output.shape)

Final Output after Self-Attention Shape: (1, 2, 64)


# Conclusion

In this notebook, we explored the foundational components of the transformer model architecture using the MLX framework, specifically tailored for Apple Silicon. Here’s a summary of what we covered:

1.	Tokenization and Embedding:

	- We began by tokenizing an example sentence into individual tokens using NLTK.
	- Each token was then converted into a unique index using a simple vocabulary.
	- We utilized an embedding layer to convert these token indices into dense vectors, preparing them for 	   further processing.
2.	Rotary Position Embedding (RoPE):

	- We applied the RoPE technique to enhance the input embeddings with positional information.
	- RoPE uses a rotational transformation based on sinusoidal functions to incorporate relative positional data directly into the embeddings.
	
3. Self-Attention Mechanism:
	- Following positional encoding, we implemented the self-attention mechanism, a core component of transformer models.
		- This included:
		- Linear transformations to obtain Query, Key, and Value matrices from the input embeddings.
		- Splitting these matrices into multiple heads for multi-head attention.
		- Calculating attention scores through scaled dot-product attention and applying the softmax function to obtain normalized attention weights.
		- Combining the output from all attention heads and applying a final linear transformation to produce the final self-attention output.